# 🪔 CropSakha AI × LeafVision — Colab Launch Edition

**☸️ Detect · Explain · Inform · Track**

This notebook runs the CropSakha full stack in Google Colab with the existing LeafVision classifier checkpoint. It reuses the trained 38-class head, starts a private model sidecar, launches FastAPI as a package, builds Next.js, and exposes only the UI through Localtunnel.

The interface keeps CropSakha's ancient Indian visual language: 🪔 diya, ☸️ dharma, 🛕 temple, 🌾 harvest, and 🌿 botanical care.

In [ ]:
# 🪔 1. Essential inputs, collected once
import getpass, os, shutil, subprocess, sys, time, secrets
from pathlib import Path

REPO_URL = os.environ.get('CROPSAKHA_REPO_URL', 'https://github.com/sumedhgurchal/CropSakha-AI.git')
REPO_BRANCH = os.environ.get('CROPSAKHA_REPO_BRANCH', 'main')
WORKDIR = Path('/content/CropSakha-AI')
LEAFVISION_DIR = Path('/content/LeafVision')
CHECKPOINT = Path('/content/CropSakha_LeafVision_classifier.pth')
BACKEND_PORT, MODEL_PORT, FRONTEND_PORT = 8000, 8001, 3000

def secret(names):
    try:
        from google.colab import userdata
        for name in names:
            value = userdata.get(name)
            if value:
                return value
    except Exception:
        pass
    return ''

GEMINI_API_KEY = secret(['GEMINI_API_KEY', 'gemini_api_key'])
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass('🔑 Gemini API key (press Enter for safe fallbacks): ')
if not CHECKPOINT.exists():
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next((name for name in uploaded if name.endswith('.pth')), None)
    if not uploaded_name:
        raise FileNotFoundError('Upload CropSakha_LeafVision_classifier.pth.')
    CHECKPOINT = Path(uploaded_name)
gemini_status = 'configured' if GEMINI_API_KEY else 'fallback prescriptions'
print('🪷 Configuration ready | Gemini:', gemini_status)
print('Repository:', REPO_URL, '| Checkpoint:', CHECKPOINT)

In [ ]:
# ☸️ 2. Install runtime packages without replacing Colab Torch
import importlib.util
def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    pip_install('torch', 'torchvision')
pip_install('fastapi==0.115.6', 'uvicorn[standard]==0.34.0', 'python-multipart==0.0.20', 'pydantic-settings==2.7.1', 'sqlalchemy==2.0.37', 'aiosqlite==0.20.0', 'python-jose[cryptography]==3.3.0', 'passlib[bcrypt]==1.7.4', 'bcrypt==4.2.1', 'email-validator==2.2.0', 'google-generativeai==0.8.4', 'requests==2.32.3', 'pillow', 'numpy', 'opencv-python-headless', 'nest-asyncio')
print('🪷 Dependencies installed')

In [ ]:
# 🌾 3. Get application and model assets
def run(command, cwd=None, check=True):
    print('$', ' '.join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=check, text=True)
if not WORKDIR.exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(WORKDIR)])
if not LEAFVISION_DIR.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/LABA-SNU/LeafVision.git', str(LEAFVISION_DIR)])
    run(['git', 'lfs', 'install', '--skip-repo'], check=False)
    run(['git', 'lfs', 'pull', '--include', 'models/LeafVision_DINO_resnet50.pth'], cwd=LEAFVISION_DIR, check=False)
LEAF_WEIGHTS = LEAFVISION_DIR / 'models' / 'LeafVision_DINO_resnet50.pth'
if not CHECKPOINT.exists() or not LEAF_WEIGHTS.exists():
    raise FileNotFoundError('Both classifier checkpoint and LeafVision_DINO_resnet50.pth are required.')
(WORKDIR / 'apps' / 'api' / 'models').mkdir(parents=True, exist_ok=True)
shutil.copy2(CHECKPOINT, WORKDIR / 'apps' / 'api' / 'models' / 'CropSakha_LeafVision_classifier.pth')
print('🌿 LeafVision backbone:', LEAF_WEIGHTS)
print('🪷 Classifier staged in the API model directory')

In [ ]:
# 🛕 4. Patch older GitHub checkouts for Colab SQLite and internal inference
models_file = WORKDIR / 'apps' / 'api' / 'models' / '__init__.py'
text = models_file.read_text(encoding='utf-8').replace('from sqlalchemy.dialects.postgresql import UUID\n', '')
if 'class UUIDType(TypeDecorator)' not in text:
    text = text.replace('from datetime import datetime\n', 'from datetime import datetime\nfrom sqlalchemy.types import CHAR, TypeDecorator\n')
    marker = 'from ..database import Base\n'
    uuid_type = '\nclass UUIDType(TypeDecorator):\n    impl = CHAR\n    cache_ok = True\n    def load_dialect_impl(self, dialect):\n        if dialect.name == postgresql:\n            from sqlalchemy.dialects.postgresql import UUID as PostgreSQLUUID\n            return dialect.type_descriptor(PostgreSQLUUID(as_uuid=True))\n        return dialect.type_descriptor(CHAR(36))\n    def process_bind_param(self, value, dialect):\n        if value is None: return None\n        value = value if isinstance(value, uuid.UUID) else uuid.UUID(str(value))\n        return value if dialect.name == postgresql else str(value)\n    def process_result_value(self, value, dialect):\n        return None if value is None else (value if isinstance(value, uuid.UUID) else uuid.UUID(str(value)))\n'
    text = text.replace(marker, marker + uuid_type).replace('UUID(as_uuid=True)', 'UUIDType()')
    models_file.write_text(text, encoding='utf-8')
config_file = WORKDIR / 'apps' / 'api' / 'config.py'
config = config_file.read_text(encoding='utf-8')
if 'inference_service_url:' not in config:
    config = config.replace('ml_model_type: str = "auto"  # auto | leafvision | mobilenet', 'ml_model_type: str = "auto"  # auto | leafvision | mobilenet\n    inference_service_url: Optional[str] = None')
    config_file.write_text(config, encoding='utf-8')
scans_file = WORKDIR / 'apps' / 'api' / 'routers' / 'scans.py'
scans_file.write_text(scans_file.read_text(encoding='utf-8').replace('colab_url = request.headers.get("x-colab-url")', 'colab_url = request.headers.get("x-colab-url") or settings.inference_service_url'), encoding='utf-8')
auth_file = WORKDIR / 'apps' / 'web' / 'src' / 'lib' / 'auth.ts'
auth_file.write_text(auth_file.read_text(encoding='utf-8').replace('fetch(`http://localhost:8000${endpoint}`, config)', 'fetch(endpoint, config)'), encoding='utf-8')
print('☸️ Compatibility patches applied')

In [ ]:
# 🧠 5. Load the reusable 38-class LeafVision classifier
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np, cv2, base64, io, asyncio, nest_asyncio
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
class_names = checkpoint.get('class_names', [])
if checkpoint.get('embed_dim') != 2048 or checkpoint.get('num_classes') != len(class_names) or not class_names:
    raise ValueError('Checkpoint metadata does not match the expected 2048-dim, 38-class model.')
backbone = models.resnet50(weights=None); backbone.fc = nn.Identity()
backbone_checkpoint = torch.load(LEAF_WEIGHTS, map_location=device, weights_only=False)
state = backbone_checkpoint.get('state_dict', backbone_checkpoint.get('model', backbone_checkpoint))
cleaned = {}
for key, value in state.items():
    clean_key = key
    for prefix in ('module.', 'backbone.', 'encoder.', 'base_encoder.', 'student.'):
        clean_key = clean_key.replace(prefix, '')
    cleaned[clean_key] = value
backbone.load_state_dict(cleaned, strict=False); backbone.to(device).eval()
classifier = nn.Linear(2048, len(class_names)).to(device)
classifier.load_state_dict(checkpoint['classifier_state_dict']); classifier.eval()
transform = transforms.Compose([transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize((0.4371, 0.5177, 0.3476), (0.1789, 0.1545, 0.1923))])
print('🪷 Model loaded on', device, '| classes:', len(class_names), '| embedding:', checkpoint['embed_dim'])

In [ ]:
# 🌿 6. Start private inference, configure the API, and launch the stack
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import JSONResponse
model_app = FastAPI(title='CropSakha LeafVision Sidecar')
def predict_image(data):
    image = Image.open(io.BytesIO(data)).convert('RGB'); tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad(): values, indices = torch.topk(F.softmax(classifier(backbone(tensor)), dim=1)[0], min(3, len(class_names)))
    top = []
    for rank, (value, index) in enumerate(zip(values.tolist(), indices.tolist()), 1):
        label = class_names[index]; parts = label.split('___', 1); crop = parts[0] if len(parts) == 2 else 'Unknown'; disease = parts[1] if len(parts) == 2 else label
        top.append({'class_name': label, 'crop': crop, 'disease': disease, 'confidence': float(value), 'rank': rank, 'is_healthy': 'healthy' in label.lower()})
    rgb = np.array(image.resize((224, 224))); green = rgb[:, :, 1].astype(float) - (rgb[:, :, 0].astype(float) + rgb[:, :, 2].astype(float)) / 2; severity = float(np.clip(np.mean(green < 15), 0, 1))
    heat = cv2.applyColorMap(np.uint8(np.clip(green.max() - green, 0, 255)), cv2.COLORMAP_JET); overlay = cv2.addWeighted(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), .62, heat, .38, 0); ok, encoded = cv2.imencode('.jpg', overlay)
    primary = top[0]; label = 'Normal' if primary['is_healthy'] else ('Mild' if severity < .3 else 'Moderate' if severity < .6 else 'Severe')
    return {'primary': primary, 'top_k': top, 'model_used': 'LeafVision DINO ResNet-50', 'severity_estimate': severity, 'severity_label': label, 'heatmap_base64': ('data:image/jpeg;base64,' + base64.b64encode(encoded).decode()) if ok else None, 'affected_area_percentage': severity * 100}
@model_app.get('/health')
async def model_health(): return {'status': 'ok', 'model_loaded': True, 'classes': len(class_names)}
@model_app.post('/predict')
async def model_predict(file: UploadFile = File(...), crop_hint: str = Form('')):
    try: return predict_image(await file.read())
    except Exception as exc: return JSONResponse(status_code=400, content={'detail': str(exc)})
nest_asyncio.apply(); import uvicorn
model_server = uvicorn.Server(uvicorn.Config(model_app, host='127.0.0.1', port=MODEL_PORT, log_level='warning')); model_task = asyncio.get_event_loop().create_task(model_server.serve()); time.sleep(2)
env_file = WORKDIR / 'apps' / 'api' / '.env'; db_file = WORKDIR / 'apps' / 'api' / 'cropsakha.db'
env_file.write_text('\n'.join(['DATABASE_URL=sqlite+aiosqlite:///' + db_file.as_posix(), 'INFERENCE_SERVICE_URL=http://127.0.0.1:' + str(MODEL_PORT), 'ML_MODEL_TYPE=leafvision', 'API_CORS_ORIGINS=*', 'JWT_SECRET_KEY=' + secrets.token_urlsafe(32), 'GEMINI_API_KEY=' + GEMINI_API_KEY, 'APP_ENV=colab']) + '\n', encoding='utf-8')
os.environ['PYTHONPATH'] = str(WORKDIR); os.environ['INFERENCE_SERVICE_URL'] = 'http://127.0.0.1:' + str(MODEL_PORT); sys.path.insert(0, str(WORKDIR))
import apps.api.main as api_module; print('☸️ Backend package import passed:', api_module.__file__)
run(['npm', 'install', '--silent'], cwd=WORKDIR / 'apps' / 'web'); run(['npm', 'run', 'build'], cwd=WORKDIR / 'apps' / 'web')
logs = WORKDIR / 'colab_logs'; logs.mkdir(exist_ok=True); backend_log = open(logs / 'backend.log', 'w'); frontend_log = open(logs / 'frontend.log', 'w')
backend = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'apps.api.main:app', '--env-file', 'apps/api/.env', '--host', '0.0.0.0', '--port', str(BACKEND_PORT)], cwd=WORKDIR, stdout=backend_log, stderr=subprocess.STDOUT, text=True)
frontend = subprocess.Popen(['npm', 'start', '--', '-p', str(FRONTEND_PORT)], cwd=WORKDIR / 'apps' / 'web', stdout=frontend_log, stderr=subprocess.STDOUT, text=True)
import urllib.request
def wait_for(url, process, name, timeout=120):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if process.poll() is not None: raise RuntimeError(name + ' exited; inspect ' + str(logs))
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200: print('✅', name, 'ready:', url); return
        except Exception: time.sleep(2)
    raise TimeoutError(name + ' did not become ready; inspect ' + str(logs))
wait_for('http://127.0.0.1:' + str(BACKEND_PORT) + '/health', backend, 'FastAPI'); wait_for('http://127.0.0.1:' + str(FRONTEND_PORT) + '/', frontend, 'Next.js')
run(['npm', 'install', '--global', 'localtunnel', '--silent']); tunnel_log = open(logs / 'localtunnel.log', 'w'); tunnel = subprocess.Popen(['lt', '--port', str(FRONTEND_PORT)], stdout=tunnel_log, stderr=subprocess.STDOUT, text=True); time.sleep(5)
print('🎉 CropSakha AI is running. UI: http://127.0.0.1:' + str(FRONTEND_PORT)); print('🌐 Localtunnel log:', logs / 'localtunnel.log'); print('🪔 Backend health passed; browser traffic uses /api.')

In [ ]:
# 🛕 7. Diagnostics and clean shutdown
def show_logs():
    for name in ('backend.log', 'frontend.log', 'localtunnel.log'):
        path = logs / name
        if path.exists(): print('\n--- ' + name + ' ---\n' + path.read_text(errors='replace')[-4000:])
def stop_all():
    for process, name in ((tunnel, 'Localtunnel'), (frontend, 'Next.js'), (backend, 'FastAPI')):
        if process and process.poll() is None: process.terminate(); print('🪔 Stopped', name)
    model_server.should_exit = True
print('Logs:', logs)
# Run show_logs() after a failure, then stop_all() when finished.